End of Phase 1
✅ Dataset loaded
✅ Shape checked
✅ Columns inspected
✅ Missing values analyzed
✅ Duplicates removed
✅ Data types verified
✅ Image availability checked
✅ Products without images removed
✅ Clean dataset saved as cleaned_styles.csv

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

In [3]:
df = pd.read_csv(r"D:\Downloads\Urika IA\data\styles.csv", on_bad_lines="skip")

In [4]:
df.head()

,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt


In [5]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 44424
Columns: 10


In [6]:
df.columns

Index(['id', 'gender', 'masterCategory', 'subCategory', 'articleType',
       'baseColour', 'season', 'year', 'usage', 'productDisplayName'],
      dtype='str')

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 44424 entries, 0 to 44423
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  44424 non-null  int64  
 1   gender              44424 non-null  str    
 2   masterCategory      44424 non-null  str    
 3   subCategory         44424 non-null  str    
 4   articleType         44424 non-null  str    
 5   baseColour          44409 non-null  str    
 6   season              44403 non-null  str    
 7   year                44423 non-null  float64
 8   usage               44107 non-null  str    
 9   productDisplayName  44417 non-null  str    
dtypes: float64(1), int64(1), str(8)
memory usage: 6.7 MB


In [8]:
df.isnull().sum()

id                      0
gender                  0
masterCategory          0
subCategory             0
articleType             0
baseColour             15
season                 21
year                    1
usage                 317
productDisplayName      7
dtype: int64

In [9]:
missing = (df.isnull().sum() / len(df)) * 100
missing.sort_values(ascending=False)

usage                 0.713578
season                0.047272
baseColour            0.033766
productDisplayName    0.015757
year                  0.002251
id                    0.000000
gender                0.000000
articleType           0.000000
masterCategory        0.000000
subCategory           0.000000
dtype: float64

In [10]:
print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 0


In [11]:
df.dtypes

id                      int64
gender                    str
masterCategory            str
subCategory               str
articleType               str
baseColour                str
season                    str
year                  float64
usage                     str
productDisplayName        str
dtype: object

In [12]:
df.describe()

,id,year
count,44424.000000,44423.000000
mean,29696.334301,2012.806497
std,17049.490518,2.126480
min,1163.000000,2007.000000
25%,14768.750000,2011.000000
50%,28618.500000,2012.000000
75%,44683.250000,2015.000000
max,60000.000000,2019.000000


In [13]:
df.describe(include="object")

,gender,masterCategory,subCategory,articleType,baseColour,season,usage,productDisplayName
count,44424,44424,44424,44424,44409,44403,44107,44417
unique,5,7,45,143,46,4,8,31121
top,Men,Apparel,Topwear,Tshirts,Black,Summer,Casual,Lucera Women Silver Earrings
freq,22147,21397,15402,7067,9728,21472,34406,82


In [14]:
import os

image_folder = r"C:\Users\abc\Downloads\images"

existing_images = 0

for pid in df["id"]:
    if os.path.exists(os.path.join(image_folder, str(pid) + ".jpg")):
        existing_images += 1

print("Images Found:", existing_images)
print("Missing Images:", len(df) - existing_images)

Images Found: 0
Missing Images: 44424


In [15]:
import os

df = df[df["id"].apply(lambda x: os.path.exists(os.path.join("images", f"{x}.jpg")))]

In [16]:
df.reset_index(drop=True, inplace=True)

In [17]:
df.to_csv("cleaned_styles.csv", index=False)

Phase 2: Feature Engineering

We'll create AI-friendly columns like:

budget_range
style
comfort_level
trend_score
body_type_match
skin_tone_match
ai_styling_tip
recommended_with
accessory_match
footwear_match

In [20]:
df = pd.read_csv(r"D:\Downloads\Urika IA\data\styles.csv", on_bad_lines="skip")

In [21]:
import numpy as np

price_map = {
    "Shirts": (800, 2500),
    "Tshirts": (400, 1200),
    "Jeans": (1000, 3000),
    "Kurtas": (1200, 3500),
    "Dresses": (1500, 5000),
    "Shoes": (1200, 6000),
    "Sandals": (800, 3000),
    "Flip Flops": (300, 1000),
    "Watches": (2000, 15000),
    "Bags": (1000, 5000)
}

np.random.seed(42)

def generate_price(article):
    low, high = price_map.get(article, (500, 2500))
    return np.random.randint(low, high)

df["price"] = df["articleType"].apply(generate_price)

In [22]:
def budget(price):
    if price < 1000:
        return "Low"
    elif price < 3000:
        return "Medium"
    elif price < 7000:
        return "High"
    else:
        return "Luxury"

df["budget_range"] = df["price"].apply(budget)

In [23]:
style_map = {
    "Shirts":"Formal",
    "Tshirts":"Casual",
    "Jeans":"Streetwear",
    "Kurtas":"Ethnic",
    "Dresses":"Party",
    "Blazers":"Formal",
    "Sweatshirts":"Sporty",
    "Jackets":"Winter Wear",
    "Shorts":"Casual"
}

df["style"] = df["articleType"].map(style_map)
df["style"] = df["style"].fillna("Casual")

In [24]:
comfort_map = {
    "Tshirts":"High",
    "Shorts":"High",
    "Jeans":"Medium",
    "Shirts":"Medium",
    "Blazers":"Low",
    "Jackets":"Medium",
    "Sweatshirts":"High"
}

df["comfort_level"] = df["articleType"].map(comfort_map)
df["comfort_level"] = df["comfort_level"].fillna("Medium")

In [25]:
def trend(year):
    if year >= 2015:
        return 10
    elif year >= 2013:
        return 8
    elif year >= 2011:
        return 6
    else:
        return 5

df["trend_score"] = df["year"].fillna(2011).apply(trend)

In [26]:
body_map = {
    "Tshirts":"Universal",
    "Shirts":"Slim",
    "Jeans":"Athletic",
    "Kurtas":"All",
    "Dresses":"Hourglass",
    "Jackets":"Rectangle"
}

df["body_type_match"] = df["articleType"].map(body_map)
df["body_type_match"] = df["body_type_match"].fillna("Universal")

In [27]:
def skin_tone(color):

    color = str(color).lower()

    if color in ["black","white","navy blue","blue"]:
        return "Universal"

    elif color in ["yellow","orange","mustard"]:
        return "Warm"

    elif color in ["pink","purple","lavender"]:
        return "Cool"

    elif color in ["green","olive"]:
        return "Neutral"

    else:
        return "Universal"

df["skin_tone_match"] = df["baseColour"].apply(skin_tone)

In [28]:
def styling_tip(row):

    return (
        f"This {row['articleType']} is perfect for "
        f"{row['usage']} occasions during {row['season']}. "
        f"Pair it with matching footwear and accessories."
    )

df["ai_styling_tip"] = df.apply(styling_tip, axis=1)

In [29]:
recommend_map = {
    "Shirts":"Jeans",
    "Tshirts":"Jeans",
    "Jeans":"T-Shirt",
    "Kurtas":"Leggings",
    "Dresses":"Heels",
    "Jackets":"Boots",
    "Sweatshirts":"Joggers"
}

df["recommended_with"] = df["articleType"].map(recommend_map)
df["recommended_with"] = df["recommended_with"].fillna("Accessories")

In [30]:
accessory_map = {
    "Shirts":"Watch",
    "Tshirts":"Cap",
    "Jeans":"Belt",
    "Kurtas":"Traditional Watch",
    "Dresses":"Handbag",
    "Jackets":"Scarf"
}

df["accessory_match"] = df["articleType"].map(accessory_map)
df["accessory_match"] = df["accessory_match"].fillna("Watch")

In [31]:
footwear_map = {
    "Formal":"Formal Shoes",
    "Casual":"Sneakers",
    "Streetwear":"Sneakers",
    "Sporty":"Sports Shoes",
    "Ethnic":"Sandals",
    "Party":"Heels",
    "Winter Wear":"Boots"
}

df["footwear_match"] = df["style"].map(footwear_map)

In [32]:
df["features"] = (
    df["gender"].fillna("") + " " +
    df["masterCategory"].fillna("") + " " +
    df["subCategory"].fillna("") + " " +
    df["articleType"].fillna("") + " " +
    df["baseColour"].fillna("") + " " +
    df["season"].fillna("") + " " +
    df["usage"].fillna("") + " " +
    df["productDisplayName"].fillna("")
)

In [33]:
df["combined_features"] = (
    df["gender"].astype(str) + " " +
    df["masterCategory"].astype(str) + " " +
    df["subCategory"].astype(str) + " " +
    df["articleType"].astype(str) + " " +
    df["baseColour"].astype(str) + " " +
    df["season"].astype(str) + " " +
    df["usage"].astype(str)
)

Phase 3: Recommendation Engine
Content-based filtering
Similarity search
Personalized recommendations

In [35]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [36]:
df = pd.read_csv(r"D:\Downloads\Urika IA\data\styles.csv", on_bad_lines="skip")
df.head()

,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt


In [37]:
df.isnull().sum()

id                      0
gender                  0
masterCategory          0
subCategory             0
articleType             0
baseColour             15
season                 21
year                    1
usage                 317
productDisplayName      7
dtype: int64

In [38]:
columns = [
    "gender",
    "masterCategory",
    "subCategory",
    "articleType",
    "baseColour",
    "season",
    "usage"
]

df[columns] = df[columns].fillna("")

In [39]:
df["features"] = (
    df["gender"].astype(str) + " " +
    df["masterCategory"].astype(str) + " " +
    df["subCategory"].astype(str) + " " +
    df["articleType"].astype(str) + " " +
    df["baseColour"].astype(str) + " " +
    df["season"].astype(str) + " " +
    df["usage"].astype(str)
)

In [40]:
df["features"] = df["features"].fillna("")

In [ ]:
import google.generativeai as genai

genai.configure(api_key="GEMINI_API_KEY")  # Replace with your actual API key

# List all models
for model in genai.list_models():
    print(model.name)
    # Check supported generation methods (optional)
    if "generateContent" in model.supported_generation_methods:
        print("  ✅ supports generateContent")
    else:
        print("  ❌ does not support generateContent")

d:\Downloads\Urika IA\fashion_style\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Aniket Pravin Redij\AppData\Local\Temp\ipykernel_12156\837796612.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


InvalidArgument: 400 API key not valid. Please pass a valid API key. [reason: "API_KEY_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "generativelanguage.googleapis.com"
}
, locale: "en-US"
message: "API key not valid. Please pass a valid API key."
]